In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as pt
import seaborn as sns
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk

In [22]:
df = pd.read_csv('./train.txt', sep=';',header=None,names=['text','emotion'])
df

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger
...,...,...
15995,i just had a very brief time in the beanbag an...,sadness
15996,i am now turning and i feel pathetic that i am...,sadness
15997,i feel strong and good overall,joy
15998,i feel like this was such a rude comment and i...,anger


In [23]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [24]:
unique_emotions = df['emotion'].unique()
emotion_numbers = {}
i = 0
for emotion in unique_emotions:
    emotion_numbers[emotion] = i
    i += 1
    
df['emotion'] = df['emotion'].map(emotion_numbers)
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [25]:
df['text'] = df['text'].apply(lambda x : x.lower())
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [26]:
def remove_punctuations(txt):
    return txt.translate(str.maketrans('','',string.punctuation))

In [27]:
df['text'] = df['text'].apply(remove_punctuations)

In [28]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new += i
    return new

df['text'] = df['text'].apply(remove_numbers)

In [29]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new


df['text'] = df['text'].apply(remove_emojis)

In [30]:
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/sharique/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/sharique/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [31]:
stop_words = set(stopwords.words('english'))

In [32]:
def remove_stopwords(txt):
    words = word_tokenize(txt)
    cleaned = []
    for i in words:
        if not i in stop_words:
            cleaned.append(i)
    return ' '.join(cleaned)

df['text'] = df['text'].apply(remove_stopwords)

In [33]:
df

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1
...,...,...
15995,brief time beanbag said anna feel like beaten,0
15996,turning feel pathetic still waiting tables sub...,0
15997,feel strong good overall,5
15998,feel like rude comment im glad,1


In [34]:
# from sklearn.feature_extraction.text import CountVectorizer

# vectorizer = CountVectorizer()

# X = vectorizer.fit_transform(df['text'])

# vocabulary = vectorizer.get_feature_names_out()

# with open('vocabulary.txt', 'w', encoding='utf-8') as file:
#     for word in vocabulary:
#         file.write(word + '\n')

# import json

# with open('vocabulary.json', 'w', encoding='utf-8') as file:
#     json.dump(vectorizer.vocabulary_, file, indent=2)

# print("Vocabulary saved to vocabulary.json")
# print(f"Saved {len(vocabulary)} words to vocabulary.txt")

In [35]:
# from sklearn.feature_extraction.text import TfidfVectorizer

# tfidf_vectorizer = TfidfVectorizer()

# X_tfidf = tfidf_vectorizer.fit_transform(df['text'])

# print("Vocabulary:", tfidf_vectorizer.get_feature_names_out())
# print("\nTF-IDF Matrix:\n", X_tfidf.toarray())

# vocabulary = tfidf_vectorizer.get_feature_names_out()

# with open('tfidf_vocabulary.txt', 'w', encoding='utf-8') as file:
#     for word in vocabulary:
#         file.write(word + '\n')

# print(f"Saved {len(vocabulary)} words to tfidf_vocabulary.txt")



In [36]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)

In [39]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score


bow_vectorizer = CountVectorizer()

X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)


nb_model = MultinomialNB()

nb_model.fit(X_train_bow, y_train)


pred_bow = nb_model.predict(X_test_bow)

print(accuracy_score(y_test, pred_bow))

0.7678125


In [40]:
tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)


nb2_model = MultinomialNB()

nb2_model.fit(X_train_tfidf, y_train)


y_pred = nb2_model.predict(X_test_tfidf)

print(accuracy_score(y_test, y_pred))

0.6609375


In [41]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(max_iter=1000)

logistic_model.fit(X_train_tfidf, y_train)

log_pred = logistic_model.predict(X_test_tfidf)

print(accuracy_score(y_test, log_pred))

0.8615625


In [42]:
from sklearn.svm import SVC

svm_model = SVC()

svm_model.fit(X_train_tfidf, y_train)

svm_pred = svm_model.predict(X_test_tfidf)

print(accuracy_score(y_test, svm_pred))

0.8515625
